# Problem Set 7

[PSet 7](MIT8_01F16_pset7.pdf)

In [1]:
import sympy as sp
import sympy.physics.mechanics as spm
import sympy.physics.vector as spv
from sympy.physics.vector.printing import init_vprinting
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


import IPython

# Import IPython display for proper LaTeX formatting
from IPython.display import display, Math, Markdown

# Initialize symbols
sp.init_printing()

# Enable dot notation printing for dynamicsymbols
init_vprinting(use_latex="mathjax")

In [2]:
def reference_frame(frame: str, x=r"\imath", y=r"\jmath", z=r"k") -> spm.ReferenceFrame:
    """Create a SymPy reference frame with custom basis vector labels.

    Parameters
    ----------
    frame : str
        The name of the reference frame.
    x, y, z : str
        Labels for the basis vectors.
    """
    return spm.ReferenceFrame(
        frame,
        latexs=(
            rf"\;{{}}^\mathcal{{{frame}}}\hat{{{x}}}",
            rf"\;{{}}^\mathcal{{{frame}}}\hat{{{y}}}",
            rf"\;{{}}^\mathcal{{{frame}}}\hat{{{z}}}",
        ),
    )


def reference_frame_circular(name: str, angle=r"theta") -> spm.ReferenceFrame:
    """Create a circular reference frame with radial and angular basis labels.

    Parameters
    ----------
    name : str
        Name of the new reference frame.
    angle : str, optional
        Symbol or label used for the angular basis vector, by default "theta".
    """
    return reference_frame(name, x=r"r", y=rf"\{angle}", z=r"e_z")

## Problem 7.1 

Object Sliding Down an Inclined Plane
An object of mass m = 4.0 kg, starting from rest, slides down an 
inclined plane of length $\ell = 3.0 \text{ m}$. The plane is inclined by 
an angle of $\theta = 30^\circ$ to the ground. The coefficient of 
kinetic friction $\mu_k = 0.2$. At the bottom of the plane, the mass slides
along a rough surface with a coefficient of kinetic friction $\mu_k = 0.3$ 
until it comes to rest. 

The goal of this problem is to find out how far the object slides along the rough
surface.

![Slide Down an Inclined Plane](../figures/PS0701-Sliding-on-a-Plane.jpg)

a. What is the work done by the friction force while the mass is sliding down the
inclined plane? (Is it positive or negative?)

b. What is the work done by the gravitational force while the mass is sliding down
the inclined plane? (Is it positive or negative?)

c. What is the kinetic energy of the mass when it just reaches the bottom of the
inclined plane?

d. Symbolically, what is the work done by the friction force while the mass is sliding
along the ground? Is this positive or negative? 

Express you answer in terms of some or all of the following: 
$m, μ_k, g, d$ where $d$ is the distance it takes the
object to stop measured from the bottom of the incline.

e. How far from the bottom of the inclined plane does the object slide along the
rough surface?


In [3]:
(
    m,  # mass of the object
    ell,  # length of the inclined plane
    theta,  # angle of the inclined plane
    μ_kip,  # coefficient of kinetic friction between the object and the inclined plane"
    μ_krp,  # coefficient of kinetic friction between the object and the rough plane"
    g,  # acceleration due to gravity
    d,  # distance it takes the object to stop measured from the bottom of the incline
) = sp.symbols("m ell theta μ_kip μ_krp g d", real=True, positive=True)

values = {
    m: sp.Rational(4, 1),  # mass of the object in kg
    theta: sp.rad(30),  # angle of the inclined plane in radians
    ell: sp.Rational(3, 1),  # length of the inclined plane in meters
    μ_kip: sp.Rational(2,10),  # coefficient of kinetic friction between the object and the inclined plane
    μ_krp: sp.Rational(3,10),  # coefficient of kinetic friction between the object and the rough plane
    g: sp.Float(9.8),  # acceleration due to gravity in m/s^2
}

In [4]:
# Newtonian reference fram at the bottom of the incline
N = reference_frame("N")

# Inclined plane reference frame. Rotated about the z-axis of the Newtonian frame 
# by an angle of pi/2 - theta
P = reference_frame("P")
P.orient_axis(N, N.z, sp.pi/2 - theta)

In [5]:
# a. Work done by friction on the object along the incline

(
    fki,  # friction force on the object from the inclined plane
    fkr,  # friction force on the object from the rough plane
    R,  # normal force on the object from the inclined plane
) = sp.symbols("f_ki f_kr R", real=True, positive=True)

y = spm.dynamicsymbols("y")  # position of the object along the inclined plane

total_force = -m * g * N.y + R * P.x + fki * P.y
display(Math(r"\text{Total force on the object on incline: }" + sp.latex(total_force)))

# Normal force and weight on the object must balance
eqn_force_block_on_incline = sp.Eq(total_force.express(P).dot(P.x), 0)

# Solve for reaction force R
R_solution = sp.solve(eqn_force_block_on_incline, R)[0]
display(
    Math(
        rf"\text{{Reaction force on the object from the incline: }} R = {sp.latex(R_solution)}"
    )
)

# Friction force on the object from the inclined plane
fki_solution = μ_kip * R_solution
display(
    Math(
        r"\text{Friction force on the object from the incline: } f_{{ki}} = "
        rf"{sp.latex(fki_solution)}"
    )
)

work_done_by_friction = sp.integrate(
    fki_solution * P.x.dot(P.x), (ell, ell, 0)  # d ell
)

display(
    Math(
        r"\text{Work done by friction on the object along the incline: } W_{{f_{{ki}}}} = "
        rf"{sp.latex(work_done_by_friction)}"
    )
)

display(
    Math(
        r"\text{Work done by friction on the object along the incline (numerical): } W_{{f_{{ki}}}}"
        rf" = {sp.latex(work_done_by_friction.subs(values).nsimplify().simplify())} "
        rf" = {sp.latex(work_done_by_friction.subs(values).evalf(4))} \text{{ Joules}}"

    )
)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [6]:
# b. Work done by gravity on the object along the incline

gravity_force = -m * g * N.y

vector_ell = ell * P.y
vector_height = vector_ell.dot(N.y)

integrand_gravity = gravity_force.dot(P.y)
display(
    Math(
        r"\text{Integrand for work done by gravity on the object along the incline: } "
        rf"F_{{gravity}} \cdot d\ell = {sp.latex(integrand_gravity)}"
    )
)
gravity_work = sp.integrate(integrand_gravity, (ell, vector_ell.dot(P.y), 0))

display(
    Math(
        r"\text{Work done by gravity on the object along the incline: } W_{{gravity}} = "
        rf"{sp.latex(gravity_work)}"
    )
)

display(
    Math(
        r"\text{Work done by gravity on the object along the incline (numerical): } W_{{gravity}} = "
        rf"{sp.latex(gravity_work.subs(values).nsimplify().simplify())} "
        rf"= {sp.latex(gravity_work.subs(values).evalf(4))} \text{{ Joules}}"
    )
)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [7]:
# c. kinetic energy of the mass when it just reaches the bottom of the
# inclined plane?

kinetic_energy_at_bottom = gravity_work + work_done_by_friction
display(
    Math(
        r"\text{Kinetic energy of the mass when it just reaches the bottom of the incline: } KE_{{bottom}} = "
        rf"{sp.latex(kinetic_energy_at_bottom)} \text{{ Joules}}"
    )
)

kinetic_energy_at_bottom_numeric = kinetic_energy_at_bottom.subs(values)
display(
    Math(
        r"\text{Kinetic energy of the mass when it just reaches the bottom of the incline (numerical): } KE_{{bottom}} = "
        rf"{sp.latex(kinetic_energy_at_bottom_numeric.nsimplify().simplify())} "
        rf"= {sp.latex(kinetic_energy_at_bottom_numeric.evalf(4))} \text{{ Joules}}"
    )
)
    

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [8]:
#d.  Work done by friction on the object along the rough plane until it stops

(
    Rrs  # normal force on the object from the rough plane
) = sp.symbols("R_rs", real=True, positive=True)

x = spm.dynamicsymbols("x")  # position of the object along the rough plane

total_force_rough_surface = -m * g * N.y + Rrs * N.y + fki * (-N.x)
display(
    Math(
        r"\text{Total force on the object on rough surface: }"  
        fr"{sp.latex(total_force_rough_surface)}"
    )
)

eqn_force_block_on_incline = sp.Eq(total_force_rough_surface.dot(N.y), 0)
display(
    Math(
        r"\text{Equation for the normal force on the object from the rough surface: }"  
        fr"{sp.latex(eqn_force_block_on_incline)}"
    )
)

rs_solution = sp.solve(eqn_force_block_on_incline, Rrs)[0]
display(
    Math(
        r"\text{Normal force on the object from the rough surface: } R_{{rs}} = "
        rf"{sp.latex(rs_solution)}"
    )
)

integrand_friction_rough_surface = -μ_krp * rs_solution
display(
    Math(
        r"\text{Integrand for work done by friction on the object along the rough surface: } "
        rf"F_{{friction}} \cdot dx = {sp.latex(integrand_friction_rough_surface)}"
    )
)

work_done_by_friction_rough_surface = sp.integrate(
    integrand_friction_rough_surface, (d, 0, d)  # d
)

display(
    Math(
        r"\text{Work done by friction on the object along the rough surface: } W_{{f_{{kr}}}} = "
        rf"{sp.latex(work_done_by_friction_rough_surface)}"
    )
)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [9]:
#e. How far from the bottom of the inclined plane does the object slide

speed_at_bottom = sp.sqrt(2 * kinetic_energy_at_bottom / m)
display(
    Math(
        r"\text{Speed of the object at the bottom of the incline: } v_{{bottom}} = "
        rf"{sp.latex(speed_at_bottom)}"
    )
)

numeric_speed_at_bottom = speed_at_bottom.subs(values)
display(
    Math(
        r"\text{Speed of the object at the bottom of the incline (numerical): } v_{{bottom}} = "
        rf"{sp.latex(numeric_speed_at_bottom.nsimplify().simplify())} "
        rf"= {sp.latex(numeric_speed_at_bottom.evalf(4))} \text{{ m/s}}"
    )
)
        
we_theorem_plane = sp.Eq(0-1/2*m*numeric_speed_at_bottom**2, work_done_by_friction_rough_surface)
we_theorem_plane_solution_d = sp.solve(we_theorem_plane, d)[0]        
we_theorem_plane_solution_d_numeric = we_theorem_plane_solution_d.subs(values)

display(
    Math(
        r"\text{Distance from the bottom of the inclined plane where the object stops (numerical): } d = "
        rf"{sp.latex(we_theorem_plane_solution_d_numeric.nsimplify().simplify())} "
        rf"= {sp.latex(we_theorem_plane_solution_d_numeric.evalf(4))} \text{{ meters}}"
    )
)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## Problem 7.2 Collision and Sliding on a Rough Surface

![](../figures/PS0702-Collison-of-Sliding-Block.jpg)

Block A of mass $m_A$ is moving horizontally with speed $v_A$ along a frictionless
surface. It collides with block B of mass $m_B$ that is initially at rest. 
The two blocks stick together after the collision. At $x = 0$, block 
B enters a rough surface with a coefficient of kinetic friction that increases 
linearly with distance $\mu_k(x) = bx$ for $0 \le x \le d$, where $b$ is a 
positive constant.

At $x = d$, block B collides with an un-stretched spring with spring constant 
$k$ on a frictionless surface. The downward gravitational acceleration 
has magnitude $g$. 

What is the distance the spring is compressed when the blocks first comes to rest? 
Express your answer in terms of some or all of the following: 

$v_A$, $b$, $d$, $g$, $k$, $m_A$ and $m_B$


![](../figures/PS0702-Collison-of-Sliding-Block.svg)

In [10]:
(
    mA,  # mass of the object
    mB,  # mass of the block
    vA,  # initial velocity of block A
    b,  # coefficient of kinetic friction constant
    d,  # distance
    g,  # gravitational acceleration
    k  # spring constant
) = sp.symbols("m_A m_B v_A b d g k", real=True, positive=True)

x = spm.dynamicsymbols("x")  # position of the block along the rough plane
muk = b*x

N = reference_frame("N")

In [11]:
# Collision between block A and block B

pAi = mA * vA # initial momentum of block A
pBi = 0  # initial momentum of block B

# Momentum conservation equation for the collision
vABf = sp.symbols("v_ABf", real=True, positive=True)  # final velocity of both blocks after collision
momentum_conservation_eqn = sp.Eq(pAi + pBi, (mA + mB) * vABf)
display(
    Math(
        r"\text{Momentum conservation equation for the collision: } "
        rf"{sp.latex(momentum_conservation_eqn)}"
    )
)   

vABf_sol = sp.solve(momentum_conservation_eqn, vABf)[0]
display(
    Math(
        r"\text{Final velocity of both blocks after collision: } v_{{ABf}} = "
        rf"{sp.latex(vABf_sol)}"
    )
)

values = {vABf: vABf_sol}

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [12]:
# N2L on the y-direction for the blocks on the rough surface

RABS = sp.symbols(
    r"R_{abs}", real=True, positive=True
)  # normal force on the blocks from the rough surface
N2L_y = sp.Eq(RABS - (mA + mB) * g, 0)
RABS_sol = sp.solve(N2L_y, RABS)[0]
values[RABS] = RABS_sol

# On rough surface after collision, work done by friction until the blocks stop
KE_initial = (sp.S.Half * (mA + mB) * vABf**2).subs(values).simplify()
display(
    Math(
        r"\text{Initial kinetic energy of the blocks after collision: } KE_{{i}} = "
        rf"\boxed{{{sp.latex(KE_initial)}}}"
    )
)

# velocity of the blocks when they reach the spring
work_done_by_friction = (
    sp.integrate(-muk * RABS, (x, 0, d)).subs(values).simplify()
)

display(
    Math(
        r"\text{Work done by friction on the blocks after collision until they stop: } W_{{f_{{kr}}}} = "
        rf"\boxed{{{sp.latex(work_done_by_friction)}}}"
    )
)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [13]:
# velocity of the blocks when they reach the spring
vd = sp.symbols("v_d", real=True, positive=True)  

# Work-energy theorem for the blocks after collision until they reach the spring
delta_KE = (sp.S.Half * (mA + mB) * vd**2 - KE_initial).simplify()

vd_sq = sp.solve(sp.Eq(delta_KE, work_done_by_friction), vd**2)[0].subs(values).simplify()
vd_sol = sp.sqrt(vd_sq)

display(
    Math(
        r"\text{Velocity of the blocks when they reach the spring: } v_{{d}} = "
        rf"\boxed{{{sp.latex(vd_sol)}}}"
    )
)

values[vd] = vd_sol

<IPython.core.display.Math object>

In [14]:
# Compression of the spring when the blocks reach it

# displacement of the spring
dl = sp.Symbol(r"\Delta \ell", real=True, positive=True)  

spring_eqn = sp.Eq(sp.S.Half*(mA+mB)*vd_sol**2, sp.S.Half * k * dl**2)
spring_eqn_sol = sp.solve(spring_eqn, dl)[0]

display(
    Math(
        r"\text{Displacement of the spring: } \Delta \ell = "
        rf"\boxed{{{sp.latex(spring_eqn_sol)}}}"
    )
)



<IPython.core.display.Math object>